In [ ]:
# Install/load
install.packages(c("MatchIt", "cobalt", "dplyr", "readr", "glue")) # run once
library(MatchIt)
library(cobalt)
library(dplyr)
library(readr)
library(glue)

In [ ]:
results <- "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-03_using_matchit_to_match_case_and_control_cohorts"

results1 <- "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-02_create_input_dataframes_for_matchit" 

In [ ]:
perform_matching <- function(input_data, 
                             match_ratio        = 10, 
                             exact_vars         = c("sex", "ancestry"),
                             plot_output_folder = NULL,   # where to save plots
                             plot_label         = NULL,   # used in title/filename
                             save_plots         = TRUE) {
  
  if (is.null(plot_label)) {
    plot_label <- "cohort"
  }
  
  m.out <- MatchIt::matchit(
    case ~ age + sex + ancestry,
    data     = input_data,
    method   = "nearest",
    exact    = exact_vars,
    ratio    = match_ratio,
    distance = "mahalanobis"
  )
  
  message("--- Matching Summary ---")
  print(summary(m.out))
  
  message("\n--- Balance Plot (Love Plot) ---")
  
  tryCatch({
    lp <- cobalt::love.plot(m.out)
    
    # Add title/subtitle if ggplot
    if (inherits(lp, "ggplot")) {
      lp <- lp +
        ggplot2::labs(
          title    = paste0("Love plot: ", plot_label),
          subtitle = paste0(match_ratio, ":1 matching (controls:cases)")
        )
    }
    
    # Show it in the R session
    print(lp)
    
    # Save to file if requested
    if (!is.null(plot_output_folder) && isTRUE(save_plots)) {
      dir.create(plot_output_folder, showWarnings = FALSE, recursive = TRUE)
      
      plot_file <- file.path(
        plot_output_folder,
        paste0("love_plot_", plot_label, ".png")
      )
      message("Saving love plot to: ", plot_file)
      
      if (inherits(lp, "ggplot")) {
        ggplot2::ggsave(
          filename = plot_file,
          plot     = lp,
          width    = 8,
          height   = 6,
          dpi      = 300
        )
      } else {
        png(plot_file, width = 1200, height = 800, res = 150)
        cobalt::love.plot(m.out)
        dev.off()
      }
    }
  }, error = function(e) {
    message("Could not generate/save love.plot. Error: ", e$message)
  })
  
  matched_data <- MatchIt::match.data(m.out)
  return(matched_data)
}


In [ ]:
batch_match_and_save <- function(input_folder,
                                 output_folder,
                                 file_pattern = "\\.csv$",
                                 ...) {
  # Create output folder if it doesn't exist
  dir.create(output_folder, showWarnings = FALSE, recursive = TRUE)
  
  # List input files
  files <- list.files(
    input_folder,
    pattern    = file_pattern,
    full.names = TRUE
  )
  
  if (length(files) == 0) {
    message("No files found in ", input_folder,
            " matching pattern: ", file_pattern)
    return(invisible(NULL))
  }
  
  for (f in files) {
    message("Processing file: ", f)
    
    # 1) Read the input data
    input_data <- read.csv(f)
    
    # 2) Base name for this file (used for both CSV and plots)
    base_name <- tools::file_path_sans_ext(basename(f))  # e.g. "cohort1"
    
    # Plot output directory for THIS file
    plot_dir <- file.path(output_folder, "love_plots", base_name)
    
    # 3) Run your matching function (pass plot info explicitly)
    matched_data <- perform_matching(
      input_data          = input_data,
      plot_output_folder  = plot_dir,
      plot_label          = base_name,
      ...
    )
    
    # 4) Build a corresponding output filename for matched data
    out_file <- file.path(
      output_folder,
      paste0("matched_", base_name, ".csv")
    )
    
    # 5) Save matched dataset
    write.csv(matched_data, out_file, row.names = FALSE)
    
    message("Saved matched data to: ", out_file, "\n")
  }
  
  invisible(NULL)
}


In [ ]:
##Function calls


batch_match_and_save(
  input_folder  = glue("{results1}/ns_matchit"),
    output_folder = glue("{results}/ns_matched")
)    
    
    
    
batch_match_and_save(
  input_folder  = glue("{results1}/b1_matchit"),
  output_folder = glue("{results}/b1_matched")
)


batch_match_and_save(
  input_folder  = glue("{results1}/ns_vax_matchit"),
  output_folder = glue("{results}/ns_vax_matched")
)


batch_match_and_save(
  input_folder  = glue("{results1}/ns_raw_vax_matchit"),
  output_folder = glue("{results}/ns_raw_vax_matched")
)



batch_match_and_save(
  input_folder  = glue("{results1}/c1_matchit"),
  output_folder = glue("{results}/c1_matched")
)



batch_match_and_save(
  input_folder  = glue("{results1}/d1_matchit"),
  output_folder = glue("{results}/d1_matched")
)




batch_match_and_save(
  input_folder  = glue("{results1}/flu_matchit"),
  output_folder = glue("{results}/flu_matched")
)




batch_match_and_save(
  input_folder  = glue("{results1}/covid_matchit"),
  output_folder = glue("{results}/covid_matched")
)


